# Spain Electricity Forecasting Benchmark

## 1. Overview

This notebook implements a complete, reproducible benchmark for one-hour-ahead Spanish electricity-load forecasting. It integrates the hourly energy and city-level weather datasets, creates leakage-safe lag, rolling, calendar, seasonal, and weather features, and compares Persistence, Linear Regression, and Random Forest on an unshuffled chronological 80/20 split.

All processed data, figures, trained models, predictions, metrics, environment details, and execution logs are generated automatically.


## 2. Dataset Description

The benchmark uses two source files from the Spain Energy Consumption dataset:

- `energy_dataset.csv`: hourly generation, forecasts, actual total load, and electricity prices.
- `weather_features.csv`: hourly weather observations for Barcelona, Bilbao, Madrid, Seville, and Valencia.

The forecasting target is **`total load actual`**, measured in MW. Weather rows may repeat for a city and timestamp when multiple weather conditions are reported; numeric values are averaged before the city-level features are pivoted to a single hourly row.

To prevent target leakage and keep the objective comparable with the PJM and UCI benchmarks, contemporaneous generation, prices, and `total load forecast` are inspected but are not used as model predictors. Predictors are restricted to historical target values, calendar/seasonal information, and numeric weather observations.


## 3. Import Libraries


In [ ]:
from datetime import datetime
from pathlib import Path
import platform
import time
import warnings

import joblib
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
    r2_score,
)

warnings.filterwarnings("default")
BENCHMARK_START_TIME = time.perf_counter()
RUN_STARTED_AT = datetime.now().astimezone()

DATETIME_COLUMN = "Datetime"
TARGET_COLUMN = "total load actual"
TRAIN_FRACTION = 0.80
N_LAGS = 24
ROLLING_WINDOWS = (24, 168)
RANDOM_STATE = 42
RF_N_ESTIMATORS = 100
RF_MAX_DEPTH = 18
RF_MIN_SAMPLES_LEAF = 2
FIGURE_DPI = 300
PREDICTION_PLOT_HOURS = 168
MODEL_COMPRESSION_LEVEL = 3

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.figsize": (12, 5),
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "legend.fontsize": 10,
    "font.size": 11,
})
print("Libraries imported and reproducibility constants initialized.")


## 4. Define Paths


In [ ]:
# The notebook is directly inside Spain/notebooks/, so its parent is Spain/.
NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name.lower() != "notebooks":
    raise RuntimeError(
        "Run this notebook from the Spain/notebooks directory. "
        f"Current working directory: {NOTEBOOK_DIR}"
    )

PROJECT_ROOT = NOTEBOOK_DIR.parent
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"

ENERGY_PATH = RAW_DATA_DIR / "energy_dataset.csv"
WEATHER_PATH = RAW_DATA_DIR / "weather_features.csv"
PROCESSED_PATH = PROCESSED_DIR / "Spain_energy_processed.csv"
RUN_LOG_PATH = RESULTS_DIR / "run_log.txt"

for directory in (PROCESSED_DIR, MODELS_DIR, RESULTS_DIR, FIGURES_DIR):
    directory.mkdir(parents=True, exist_ok=True)

LOG_MESSAGES = []


def log_event(message):
    """Display and immediately persist a timestamped run-log entry."""
    stamp = datetime.now().astimezone().isoformat(timespec="seconds")
    entry = f"[{stamp}] {message}"
    LOG_MESSAGES.append(entry)
    RUN_LOG_PATH.write_text("\n".join(LOG_MESSAGES) + "\n", encoding="utf-8")
    print(entry)


def save_figure(filename):
    """Save the active figure with consistent publication settings."""
    path = FIGURES_DIR / filename
    plt.tight_layout()
    plt.savefig(path, dpi=FIGURE_DPI, bbox_inches="tight")
    plt.show()
    plt.close()
    log_event(f"Figure saved: {filename}")


def save_model(model, destination):
    """Compress a model to a temporary file and publish it atomically."""
    temporary = destination.with_name(f"{destination.name}.tmp")
    if temporary.exists():
        temporary.unlink()
    try:
        joblib.dump(model, temporary, compress=("gzip", MODEL_COMPRESSION_LEVEL))
        temporary.replace(destination)
    except Exception:
        if temporary.exists():
            temporary.unlink()
        raise


log_event(f"Benchmark started: {RUN_STARTED_AT.isoformat(timespec='seconds')}")
log_event(f"Project root: {PROJECT_ROOT}")
log_event(f"Python {platform.python_version()} on {platform.platform()}")


## 5. Load Data


In [ ]:
missing_files = [path for path in (ENERGY_PATH, WEATHER_PATH) if not path.is_file()]
if missing_files:
    raise FileNotFoundError(
        "Required raw dataset file(s) not found:\n- "
        + "\n- ".join(str(path) for path in missing_files)
    )

energy_raw = pd.read_csv(ENERGY_PATH, low_memory=False)
weather_raw = pd.read_csv(WEATHER_PATH, low_memory=False)
ENERGY_ORIGINAL_ROWS = len(energy_raw)
WEATHER_ORIGINAL_ROWS = len(weather_raw)

log_event(f"Energy dataset loaded: {energy_raw.shape[0]:,} rows × {energy_raw.shape[1]} columns")
log_event(f"Weather dataset loaded: {weather_raw.shape[0]:,} rows × {weather_raw.shape[1]} columns")
display(energy_raw.head())
display(weather_raw.head())


## 6. Data Exploration


In [ ]:
required_energy_columns = {"time", TARGET_COLUMN}
required_weather_columns = {"dt_iso", "city_name"}
missing_energy = required_energy_columns.difference(energy_raw.columns)
missing_weather = required_weather_columns.difference(weather_raw.columns)
if missing_energy or missing_weather:
    raise ValueError(
        f"Missing energy columns: {sorted(missing_energy)}; "
        f"missing weather columns: {sorted(missing_weather)}"
    )

print("ENERGY DATA")
print("Shape:", energy_raw.shape)
display(energy_raw.isna().sum().sort_values(ascending=False).to_frame("Missing"))
display(energy_raw.describe(include="all"))

print("\nWEATHER DATA")
print("Shape:", weather_raw.shape)
print("Cities:", sorted(weather_raw["city_name"].astype(str).str.strip().unique()))
display(weather_raw.isna().sum().sort_values(ascending=False).to_frame("Missing"))
display(weather_raw.describe(include="all"))

energy_preview_time = pd.to_datetime(energy_raw["time"], utc=True, errors="coerce")
plt.figure(figsize=(14, 5))
plt.plot(energy_preview_time, energy_raw[TARGET_COLUMN], linewidth=0.55, color="#1f77b4")
plt.title("Spain Actual Electricity Load")
plt.xlabel("Datetime (UTC)")
plt.ylabel("Actual Total Load (MW)")
save_figure("time_series.png")

plt.figure(figsize=(10, 5))
plt.hist(energy_raw[TARGET_COLUMN].dropna(), bins=50, color="#4c78a8", edgecolor="white")
plt.title("Distribution of Spain Actual Electricity Load")
plt.xlabel("Actual Total Load (MW)")
plt.ylabel("Frequency")
save_figure("data_distribution.png")
log_event("Initial data exploration completed")


## 7. Data Cleaning

Timestamps are normalized to UTC so daylight-saving offsets cannot create false mismatches. The 36 missing target observations are filled by time interpolation to preserve the regular hourly sequence. Weather city names are stripped, duplicate city–timestamp records are averaged, and physically invalid pressure and wind-speed values are treated as missing before time interpolation within each city.


In [ ]:
energy = energy_raw.copy()
energy[DATETIME_COLUMN] = pd.to_datetime(energy["time"], utc=True, errors="coerce")
energy[TARGET_COLUMN] = pd.to_numeric(energy[TARGET_COLUMN], errors="coerce")
energy = energy.dropna(subset=[DATETIME_COLUMN]).sort_values(DATETIME_COLUMN).reset_index(drop=True)

if energy[DATETIME_COLUMN].duplicated().any():
    raise ValueError("Energy data contains duplicate UTC timestamps; one-to-one integration is unsafe")

target_missing_before = int(energy[TARGET_COLUMN].isna().sum())
energy = energy.set_index(DATETIME_COLUMN)
energy[TARGET_COLUMN] = energy[TARGET_COLUMN].interpolate(method="time", limit_direction="both")
energy = energy.reset_index()
if energy[TARGET_COLUMN].isna().any():
    raise ValueError("Target still contains missing values after interpolation")

weather = weather_raw.copy()
weather[DATETIME_COLUMN] = pd.to_datetime(weather["dt_iso"], utc=True, errors="coerce")
weather["city_name"] = weather["city_name"].astype(str).str.strip()
weather = weather.dropna(subset=[DATETIME_COLUMN, "city_name"])

WEATHER_VARIABLES = [
    "temp", "temp_min", "temp_max", "pressure", "humidity",
    "wind_speed", "wind_deg", "rain_1h", "rain_3h", "snow_3h", "clouds_all",
]
for column in WEATHER_VARIABLES:
    weather[column] = pd.to_numeric(weather[column], errors="coerce")

# Remove clear sensor/data-entry outliers before city-wise interpolation.
weather.loc[~weather["pressure"].between(850, 1100), "pressure"] = np.nan
weather.loc[~weather["wind_speed"].between(0, 75), "wind_speed"] = np.nan

duplicate_weather_rows = int(weather.duplicated([DATETIME_COLUMN, "city_name"], keep=False).sum())
weather_hourly = (
    weather.groupby([DATETIME_COLUMN, "city_name"], as_index=False)[WEATHER_VARIABLES]
    .mean()
    .sort_values(["city_name", DATETIME_COLUMN])
)
weather_hourly[WEATHER_VARIABLES] = (
    weather_hourly.groupby("city_name")[WEATHER_VARIABLES]
    .transform(lambda values: values.interpolate(limit_direction="both"))
)

log_event(
    f"Data cleaning completed: {target_missing_before} target values interpolated; "
    f"{duplicate_weather_rows:,} rows participating in duplicate city-time records aggregated"
)


## 8. Dataset Integration


In [ ]:
weather_wide = weather_hourly.pivot(
    index=DATETIME_COLUMN,
    columns="city_name",
    values=WEATHER_VARIABLES,
)
weather_wide.columns = [
    f"weather_{variable}_{city.lower().replace(' ', '_')}"
    for variable, city in weather_wide.columns
]
weather_wide = weather_wide.reset_index()
WEATHER_FEATURE_COLUMNS = weather_wide.columns.drop(DATETIME_COLUMN).tolist()

integrated_df = energy[[DATETIME_COLUMN, TARGET_COLUMN]].merge(
    weather_wide,
    on=DATETIME_COLUMN,
    how="inner",
    validate="one_to_one",
)
integrated_df = integrated_df.sort_values(DATETIME_COLUMN).reset_index(drop=True)
integrated_df[WEATHER_FEATURE_COLUMNS] = integrated_df[WEATHER_FEATURE_COLUMNS].interpolate(
    limit_direction="both"
)

missing_weather_columns = integrated_df[WEATHER_FEATURE_COLUMNS].columns[
    integrated_df[WEATHER_FEATURE_COLUMNS].isna().any()
].tolist()
if missing_weather_columns:
    raise ValueError(f"Weather features still contain missing values: {missing_weather_columns}")
if integrated_df.empty:
    raise ValueError("Energy/weather integration produced no observations")

# Compact relationship view: target versus mean weather conditions across cities.
relationship_df = integrated_df[[TARGET_COLUMN]].copy()
for variable in ("temp", "pressure", "humidity", "wind_speed", "clouds_all"):
    columns = [c for c in WEATHER_FEATURE_COLUMNS if c.startswith(f"weather_{variable}_")]
    relationship_df[f"mean_{variable}"] = integrated_df[columns].mean(axis=1)
correlation = relationship_df.corr()

plt.figure(figsize=(8, 7))
image = plt.imshow(correlation, interpolation="nearest", cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar(image, label="Pearson correlation")
plt.xticks(range(len(correlation)), correlation.columns, rotation=55, ha="right")
plt.yticks(range(len(correlation)), correlation.columns)
plt.title("Load and Weather Correlation Matrix")
save_figure("correlation_heatmap.png")
display(correlation)

log_event(
    f"Datasets integrated one-to-one: {len(integrated_df):,} hourly rows and "
    f"{len(WEATHER_FEATURE_COLUMNS)} city-specific weather features"
)


## 9. Feature Engineering


In [ ]:
def create_features(frame):
    """Create leakage-safe historical, rolling, calendar, and seasonal features."""
    featured = frame.copy()
    for lag in range(1, N_LAGS + 1):
        featured[f"lag_{lag}"] = featured[TARGET_COLUMN].shift(lag)

    historical_target = featured[TARGET_COLUMN].shift(1)
    for window in ROLLING_WINDOWS:
        featured[f"rolling_mean_{window}"] = historical_target.rolling(window).mean()
        featured[f"rolling_std_{window}"] = historical_target.rolling(window).std()

    timestamps = featured[DATETIME_COLUMN].dt
    featured["hour"] = timestamps.hour
    featured["day"] = timestamps.day
    featured["weekday"] = timestamps.dayofweek
    featured["month"] = timestamps.month
    featured["dayofyear"] = timestamps.dayofyear
    featured["weekofyear"] = timestamps.isocalendar().week.astype("int16")
    featured["season"] = ((timestamps.month % 12) // 3).astype("int8")
    featured["is_weekend"] = (timestamps.dayofweek >= 5).astype("int8")
    featured["hour_sin"] = np.sin(2 * np.pi * timestamps.hour / 24)
    featured["hour_cos"] = np.cos(2 * np.pi * timestamps.hour / 24)
    featured["year_sin"] = np.sin(2 * np.pi * timestamps.dayofyear / 365.25)
    featured["year_cos"] = np.cos(2 * np.pi * timestamps.dayofyear / 365.25)
    return featured.dropna().reset_index(drop=True)


model_df = create_features(integrated_df)
HISTORICAL_FEATURES = [f"lag_{lag}" for lag in range(1, N_LAGS + 1)] + [
    f"rolling_{stat}_{window}"
    for window in ROLLING_WINDOWS
    for stat in ("mean", "std")
]
CALENDAR_FEATURES = [
    "hour", "day", "weekday", "month", "dayofyear", "weekofyear",
    "season", "is_weekend", "hour_sin", "hour_cos", "year_sin", "year_cos",
]
FEATURE_COLUMNS = HISTORICAL_FEATURES + CALENDAR_FEATURES + WEATHER_FEATURE_COLUMNS

if model_df.empty or model_df[FEATURE_COLUMNS + [TARGET_COLUMN]].isna().any().any():
    raise ValueError("Feature engineering produced an empty or incomplete modeling dataset")

model_df.to_csv(PROCESSED_PATH, index=False)
if PROCESSED_PATH.stat().st_size == 0:
    raise IOError(f"Processed dataset was not saved correctly: {PROCESSED_PATH}")

print(f"Processed rows: {len(model_df):,}")
print(f"Total predictors: {len(FEATURE_COLUMNS)}")
print(f"Historical: {len(HISTORICAL_FEATURES)}; calendar: {len(CALENDAR_FEATURES)}; weather: {len(WEATHER_FEATURE_COLUMNS)}")
display(model_df.head())
log_event(f"Feature engineering completed: {len(model_df):,} rows × {len(FEATURE_COLUMNS)} predictors")


## 10. Train/Test Split


In [ ]:
split_index = int(len(model_df) * TRAIN_FRACTION)
if split_index <= 0 or split_index >= len(model_df):
    raise ValueError("Chronological split requires non-empty training and testing sets")

train_df = model_df.iloc[:split_index].copy()
test_df = model_df.iloc[split_index:].copy()
X_train = train_df[FEATURE_COLUMNS]
y_train = train_df[TARGET_COLUMN]
X_test = test_df[FEATURE_COLUMNS]
y_test = test_df[TARGET_COLUMN]

print(f"Training samples: {len(train_df):,}")
print(f"Testing samples: {len(test_df):,}")
print(f"Train: {train_df[DATETIME_COLUMN].iloc[0]} to {train_df[DATETIME_COLUMN].iloc[-1]}")
print(f"Test:  {test_df[DATETIME_COLUMN].iloc[0]} to {test_df[DATETIME_COLUMN].iloc[-1]}")

plt.figure(figsize=(14, 5))
plt.plot(train_df[DATETIME_COLUMN], y_train, label="Training data", linewidth=0.5)
plt.plot(test_df[DATETIME_COLUMN], y_test, label="Testing data", linewidth=0.5)
plt.axvline(test_df[DATETIME_COLUMN].iloc[0], color="black", linestyle="--", label="80/20 split")
plt.title("Chronological Train/Test Split")
plt.xlabel("Datetime (UTC)")
plt.ylabel("Actual Total Load (MW)")
plt.legend()
save_figure("train_test_split.png")
log_event("Chronological 80/20 train/test split completed without shuffling")


## 11. Baseline Model


In [ ]:
def evaluate_predictions(model_name, actual, predicted, training_seconds):
    """Calculate the common benchmark metrics."""
    actual_array = np.asarray(actual, dtype=float)
    predicted_array = np.asarray(predicted, dtype=float)
    if actual_array.shape != predicted_array.shape:
        raise ValueError(f"Prediction shape mismatch for {model_name}")
    return {
        "Model": model_name,
        "MAE": mean_absolute_error(actual_array, predicted_array),
        "RMSE": mean_squared_error(actual_array, predicted_array) ** 0.5,
        "MAPE": mean_absolute_percentage_error(actual_array, predicted_array) * 100,
        "R2": r2_score(actual_array, predicted_array),
        "Training_Time_Seconds": float(training_seconds),
    }


metrics_records = []
prediction_store = {}
start = time.perf_counter()
persistence_predictions = X_test["lag_1"].to_numpy()
persistence_time = time.perf_counter() - start
persistence_metrics = evaluate_predictions(
    "Persistence", y_test, persistence_predictions, persistence_time
)
metrics_records.append(persistence_metrics)
prediction_store["Persistence_Prediction"] = persistence_predictions
display(pd.DataFrame([persistence_metrics]).round(4))
log_event(f"Persistence evaluated: RMSE={persistence_metrics['RMSE']:.4f} MW")


## 12. Linear Regression


In [ ]:
linear_regression = LinearRegression()
start = time.perf_counter()
linear_regression.fit(X_train, y_train)
linear_training_time = time.perf_counter() - start
linear_predictions = linear_regression.predict(X_test)
linear_metrics = evaluate_predictions(
    "Linear Regression", y_test, linear_predictions, linear_training_time
)
metrics_records.append(linear_metrics)
prediction_store["Linear_Regression_Prediction"] = linear_predictions
display(pd.DataFrame([linear_metrics]).round(4))
log_event(
    f"Linear Regression trained in {linear_training_time:.3f}s; "
    f"RMSE={linear_metrics['RMSE']:.4f} MW"
)


## 13. Random Forest


In [ ]:
random_forest = RandomForestRegressor(
    n_estimators=RF_N_ESTIMATORS,
    max_depth=RF_MAX_DEPTH,
    min_samples_leaf=RF_MIN_SAMPLES_LEAF,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
start = time.perf_counter()
random_forest.fit(X_train, y_train)
rf_training_time = time.perf_counter() - start
rf_predictions = random_forest.predict(X_test)
rf_metrics = evaluate_predictions(
    "Random Forest", y_test, rf_predictions, rf_training_time
)
metrics_records.append(rf_metrics)
prediction_store["Random_Forest_Prediction"] = rf_predictions
display(pd.DataFrame([rf_metrics]).round(4))
log_event(
    f"Random Forest trained in {rf_training_time:.3f}s; "
    f"RMSE={rf_metrics['RMSE']:.4f} MW"
)


## 14. Model Evaluation


In [ ]:
results_df = pd.DataFrame(metrics_records).sort_values("RMSE", ignore_index=True)
display(results_df.style.format({
    "MAE": "{:.2f}", "RMSE": "{:.2f}", "MAPE": "{:.3f}%",
    "R2": "{:.4f}", "Training_Time_Seconds": "{:.3f}",
}))

plot_count = min(PREDICTION_PLOT_HOURS, len(test_df))
plt.figure(figsize=(14, 6))
plt.plot(test_df[DATETIME_COLUMN].iloc[:plot_count], y_test.iloc[:plot_count], label="Actual", linewidth=2)
plt.plot(test_df[DATETIME_COLUMN].iloc[:plot_count], persistence_predictions[:plot_count], label="Persistence", linewidth=1)
plt.plot(test_df[DATETIME_COLUMN].iloc[:plot_count], linear_predictions[:plot_count], label="Linear Regression", linewidth=1)
plt.plot(test_df[DATETIME_COLUMN].iloc[:plot_count], rf_predictions[:plot_count], label="Random Forest", linewidth=1)
plt.title("Spain Load Forecasts — First Test Week")
plt.xlabel("Datetime (UTC)")
plt.ylabel("Actual Total Load (MW)")
plt.legend(ncol=2)
save_figure("prediction_comparison.png")

plt.figure(figsize=(9, 5))
bars = plt.bar(results_df["Model"], results_df["RMSE"], color=["#4c78a8", "#f58518", "#54a24b"])
plt.bar_label(bars, fmt="%.1f", padding=3)
plt.title("Spain Benchmark Model Comparison")
plt.xlabel("Model")
plt.ylabel("RMSE (MW; lower is better)")
save_figure("model_comparison.png")
log_event(f"Model evaluation completed; best model by RMSE: {results_df.iloc[0]['Model']}")


## 15. Save Outputs


In [ ]:
RESULTS_PATH = RESULTS_DIR / "benchmark_results.csv"
PREDICTIONS_PATH = RESULTS_DIR / "predictions.csv"
METRICS_PATH = RESULTS_DIR / "metrics.txt"
ENVIRONMENT_PATH = RESULTS_DIR / "environment.txt"
LINEAR_MODEL_PATH = MODELS_DIR / "linear_regression_model.pkl"
RF_MODEL_PATH = MODELS_DIR / "random_forest_model.pkl"

save_model(linear_regression, LINEAR_MODEL_PATH)
save_model(random_forest, RF_MODEL_PATH)
results_df.to_csv(RESULTS_PATH, index=False)

predictions_df = pd.DataFrame({
    DATETIME_COLUMN: test_df[DATETIME_COLUMN].to_numpy(),
    "Actual": y_test.to_numpy(),
    **prediction_store,
})
predictions_df.to_csv(PREDICTIONS_PATH, index=False)

metric_lines = ["Spain Electricity Forecasting Benchmark", "=" * 80]
for record in metrics_records:
    metric_lines.extend([
        "", f"Model: {record['Model']}",
        f"MAE: {record['MAE']:.6f}", f"RMSE: {record['RMSE']:.6f}",
        f"MAPE: {record['MAPE']:.6f}%", f"R²: {record['R2']:.6f}",
        f"Training time: {record['Training_Time_Seconds']:.6f} seconds",
    ])
METRICS_PATH.write_text("\n".join(metric_lines) + "\n", encoding="utf-8")

TOTAL_RUNTIME = time.perf_counter() - BENCHMARK_START_TIME
environment_lines = [
    f"Python version: {platform.python_version()}",
    f"Platform: {platform.platform()}",
    f"pandas version: {pd.__version__}",
    f"numpy version: {np.__version__}",
    f"matplotlib version: {matplotlib.__version__}",
    f"scikit-learn version: {sklearn.__version__}",
    f"joblib version: {joblib.__version__}",
    "xgboost version: not used",
    f"Working directory: {NOTEBOOK_DIR}",
    f"Benchmark runtime: {TOTAL_RUNTIME:.6f} seconds",
]
ENVIRONMENT_PATH.write_text("\n".join(environment_lines) + "\n", encoding="utf-8")

for record in metrics_records:
    log_event(
        f"Final metrics — {record['Model']}: MAE={record['MAE']:.6f}, "
        f"RMSE={record['RMSE']:.6f}, MAPE={record['MAPE']:.6f}%, "
        f"R2={record['R2']:.6f}, training={record['Training_Time_Seconds']:.3f}s"
    )
log_event("Models trained: Persistence, Linear Regression, Random Forest")
log_event(f"Total benchmark runtime: {TOTAL_RUNTIME:.3f} seconds")

required_artifacts = [
    PROCESSED_PATH, LINEAR_MODEL_PATH, RF_MODEL_PATH, RESULTS_PATH,
    PREDICTIONS_PATH, METRICS_PATH, ENVIRONMENT_PATH, RUN_LOG_PATH,
]
required_figures = [
    FIGURES_DIR / "time_series.png", FIGURES_DIR / "data_distribution.png",
    FIGURES_DIR / "correlation_heatmap.png", FIGURES_DIR / "train_test_split.png",
    FIGURES_DIR / "prediction_comparison.png", FIGURES_DIR / "model_comparison.png",
]
errors = [
    f"Missing or empty artifact: {path}"
    for path in required_artifacts + required_figures
    if not path.is_file() or path.stat().st_size == 0
]
if len(pd.read_csv(RESULTS_PATH)) != 3:
    errors.append("benchmark_results.csv must contain exactly three model rows")
expected_prediction_columns = [
    DATETIME_COLUMN, "Actual", "Persistence_Prediction",
    "Linear_Regression_Prediction", "Random_Forest_Prediction",
]
if pd.read_csv(PREDICTIONS_PATH, nrows=0).columns.tolist() != expected_prediction_columns:
    errors.append("predictions.csv contains incorrect columns")
if pd.read_csv(PROCESSED_PATH, nrows=1).empty:
    errors.append("Processed feature dataset is empty")
if errors:
    raise RuntimeError("Final validation failed:\n- " + "\n- ".join(errors))

log_event("Final artifact validation passed")
print("FINAL VALIDATION PASSED")
print(f"Artifacts saved under: {PROJECT_ROOT}")


## 16. Conclusions


In [ ]:
best = results_df.iloc[0]
print("SPAIN BENCHMARK SUMMARY")
print("=" * 80)
print(f"Original energy rows: {ENERGY_ORIGINAL_ROWS:,}")
print(f"Original weather rows: {WEATHER_ORIGINAL_ROWS:,}")
print(f"Integrated hourly rows: {len(integrated_df):,}")
print(f"Processed modeling rows: {len(model_df):,}")
print(f"Predictors: {len(FEATURE_COLUMNS)}")
print(f"Training samples: {len(train_df):,}")
print(f"Testing samples: {len(test_df):,}")
print(f"Best model by RMSE: {best['Model']}")
print(f"Best MAE: {best['MAE']:.6f} MW")
print(f"Best RMSE: {best['RMSE']:.6f} MW")
print(f"Best MAPE: {best['MAPE']:.6f}%")
print(f"Best R²: {best['R2']:.6f}")
print(f"Total runtime: {TOTAL_RUNTIME:.3f} seconds")
display(results_df)


The persistence model provides the essential one-hour-ahead reference. Linear Regression measures the value of the engineered predictors under a linear relationship, while Random Forest captures nonlinear interactions among recent demand, seasonal patterns, and city-level weather. The best-performing model is selected dynamically from the untouched chronological test period using the lowest RMSE; no model outcome is hardcoded.

This benchmark is directly comparable in structure and evaluation practice with the PJM and UCI Household benchmark packages.
